In [23]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from IPython import display

In [24]:
env = gym.make("CartPole-v1", render_mode="human")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [25]:
env.metadata['render_fps'] = 80

In [26]:
env.metadata

{'render_modes': ['human', 'rgb_array'], 'render_fps': 80}

In [27]:
Transition = namedtuple(
    "Transition",
    ("state", "action", "next_state", "reward")
)

In [28]:
class ReplayMemory(object):

    #initializer
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)



    #save transition
    def push(self, *args):
        self.memory.append(Transition(*args))

    #random sample
    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)



    #memory length
    def __len__(self):
        return len(self.memory)



In [29]:
class DQN(nn.Module):

    def __init__(self, num_observation, num_action):
        super(DQN, self).__init__()

        self.layer1 = nn.Linear(in_features=num_observation, out_features=128)
        self.layer2 = nn.Linear(in_features=128, out_features=128)
        self.layer3 = nn.Linear(in_features=128, out_features=num_action)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = self.layer3(x)
        return x

In [30]:
batch_size = 128
gamma = 0.9 # discount factor
epsilon_start = 0.9
epsilon_end = 0.05
epsilon_decay = 1000
tau = 0.005 # update rate of target
learning_rate = 1e-4

In [31]:
num_action = env.action_space.n
num_action

2

In [32]:
state, info = env.reset()
print(state)   # initial reset 
print(info)

[-0.04737483  0.02359744  0.00719819  0.02380812]
{}


In [33]:
num_observation = len(state)
num_observation

4

In [34]:
policy_net =DQN(num_observation=num_observation, num_action=num_action).to(device)
policy_net

DQN(
  (layer1): Linear(in_features=4, out_features=128, bias=True)
  (layer2): Linear(in_features=128, out_features=128, bias=True)
  (layer3): Linear(in_features=128, out_features=2, bias=True)
)

In [35]:
target_net = DQN(num_observation=num_observation, num_action=num_action).to(device)
target_net

DQN(
  (layer1): Linear(in_features=4, out_features=128, bias=True)
  (layer2): Linear(in_features=128, out_features=128, bias=True)
  (layer3): Linear(in_features=128, out_features=2, bias=True)
)

In [36]:
policy_net.state_dict()

OrderedDict([('layer1.weight',
              tensor([[-0.4044,  0.0200, -0.4984, -0.1837],
                      [ 0.2429,  0.0926, -0.4693,  0.1388],
                      [-0.0501,  0.0563, -0.0800, -0.2875],
                      [-0.3735,  0.2873,  0.0942,  0.1927],
                      [-0.0021,  0.4857,  0.4745,  0.0352],
                      [-0.2192,  0.3214, -0.0378,  0.3104],
                      [-0.2258, -0.3315, -0.2748,  0.1449],
                      [ 0.1443, -0.1689, -0.4895,  0.3936],
                      [ 0.4414, -0.1220, -0.4612,  0.1899],
                      [ 0.2883,  0.3910,  0.3259, -0.1651],
                      [ 0.2762, -0.1048, -0.1076, -0.0500],
                      [-0.2008,  0.1059, -0.4525, -0.2056],
                      [-0.3217, -0.2831,  0.1187,  0.4383],
                      [-0.0367,  0.3552, -0.3252, -0.2384],
                      [-0.1860,  0.3380, -0.0159, -0.1294],
                      [ 0.4828,  0.3611, -0.2725,  0.1017],
         

In [37]:
target_net.load_state_dict(policy_net.state_dict())
target_net.state_dict()

OrderedDict([('layer1.weight',
              tensor([[-0.4044,  0.0200, -0.4984, -0.1837],
                      [ 0.2429,  0.0926, -0.4693,  0.1388],
                      [-0.0501,  0.0563, -0.0800, -0.2875],
                      [-0.3735,  0.2873,  0.0942,  0.1927],
                      [-0.0021,  0.4857,  0.4745,  0.0352],
                      [-0.2192,  0.3214, -0.0378,  0.3104],
                      [-0.2258, -0.3315, -0.2748,  0.1449],
                      [ 0.1443, -0.1689, -0.4895,  0.3936],
                      [ 0.4414, -0.1220, -0.4612,  0.1899],
                      [ 0.2883,  0.3910,  0.3259, -0.1651],
                      [ 0.2762, -0.1048, -0.1076, -0.0500],
                      [-0.2008,  0.1059, -0.4525, -0.2056],
                      [-0.3217, -0.2831,  0.1187,  0.4383],
                      [-0.0367,  0.3552, -0.3252, -0.2384],
                      [-0.1860,  0.3380, -0.0159, -0.1294],
                      [ 0.4828,  0.3611, -0.2725,  0.1017],
         

In [38]:
optimizer = optim.AdamW(params=policy_net.parameters(), lr=learning_rate)
memory = ReplayMemory(10000)
optimizer

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0.01
)

In [39]:
steps_done = 0 

In [40]:
def select_action(state):
    global steps_done
    sample = random.random() # 0 t0 1 random number
    epsilon_threshold = epsilon_end + (epsilon_start - epsilon_end) * math.exp(-1 * steps_done / epsilon_decay)
    steps_done += 1

    # if sample > threshold action is selected by neural network else random action is selected
    if sample > epsilon_threshold:
        with torch.no_grad():
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)


In [41]:
episode_duration = []

In [42]:
def plot_duration(show_result = False):

    plt.figure(1)
    duration_t = torch.tensor(episode_duration, dtype=torch.float)
    if show_result:
        plt.title("Result")
    else:
        plt.clf()
        plt.title("Training")

    plt.xlabel("Episode")
    plt.ylabel("Duration")
    plt.plot(duration_t.numpy())

    if len(duration_t) > 100:
        means = duration_t.unfold(0,100,1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)

    display.display(plt.gcf())
    display.clear_output(wait=True)

In [43]:
def optimize_model():
    #chef if tehre is enough experience in memory
    if len(memory) < batch_size:
        return
    
    # take random sample from memory
    transition = memory.sample(batch_size=batch_size)


    batch = Transition(*zip(*transition))

    #boolean mask for next_state is not None
    non_final_mask = torch.tensor(tuple(map(lambda s:s is not None, batch.next_state)), device=device, dtype=torch.bool)
    
    non_final_next = torch.cat([s for s in batch.next_state if s is not None])

    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Compute V(s_{t+1}) for all next states.
    next_state_values = torch.zeros(batch_size, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next).max(1).values
    # Compute the expected Q values
    expected_state_action_values = (next_state_values * gamma) + reward_batch

    # Compute Huber loss
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    # In-place gradient clipping
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()


In [ ]:
num_episodes = 250

for i_episode in range(num_episodes):

    state, info = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    for t in count():
        action = select_action(state)
        observation, reward, terminated, truncated, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        done = terminated or truncated ## lose

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # store transition in memory
        memory.push(state, action, next_state, reward)

        state = next_state

        #training
        optimize_model()

        #update network parameters
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()


        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*tau + target_net_state_dict[key]*(1-tau)
        target_net.load_state_dict(target_net_state_dict)


        if done:
            episode_duration.append(t + 1)
            plot_duration()
            break

print("Done")
plot_duration(show_result=True)
plt.ioff()
plt.show()